# Teste de Pipeline de Coleta de Dados do FIRMS

In [30]:
# importando bilbiotecas
import geopandas as gpd
from pathlib import Path
import pandas as pd
import folium
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

In [31]:
# configurações de iniciais
BASE_DIR = Path(Path.cwd()).resolve().parent
load_dotenv()

True

#### 1. Filtragem dos dados falsos de queimadas pelo polígono do Parque Nacional da Chapada Diamantina

In [11]:
#Coletar o bounding box da chapada
def get_bbox(kml=BASE_DIR / 'data' / 'shapefile' / 'PARNA_Chap_Diamantina.kml'):

    coord = gpd.read_file(kml)

    sw = [coord.total_bounds[1], coord.total_bounds[0]]
    ne = [coord.total_bounds[3], coord.total_bounds[2]]

    return sw, ne

# Mapa destacando o bounding box da Chapada Diamantina
sw,ne = get_bbox()

center_lat = (sw[0] + ne[0]) / 2
center_lon = (sw[1] + ne[1]) / 2

In [3]:
def load_boundary(kml):
    """
    Carrega o arquivo KML do Parque Nacional da Chapada Diamantina e retorna um GeoDataFrame.
    """
    gdf = gpd.read_file(kml).union_all()
    print(f"Polígono carregado do arquivo KML")
    return gdf


def filter_parna_chapada(df,
                      kml=BASE_DIR / 'data' / 'shapefile' / 'PARNA_Chap_Diamantina.kml'):
    """
    Filtra os dados de queimadas para incluir apenas aqueles dentro do polígono.
    """

    #Carrega o poligono
    polygons = load_boundary(kml)

    # Converte o df para GeoDataFrame
    gdf = gpd.GeoDataFrame(df,
                           geometry=gpd.points_from_xy(df['longitude'],df['latitude']),
                           crs="EPSG:4326")

    # Filtra os pontos que estão dentro do polígono
    gdf_filtered = gdf.geometry.within(polygons)
    df_filtred = gdf[gdf_filtered].drop(columns='geometry')

    return df_filtred





In [7]:
# carregando dados falsos de queimadas
df_raw = pd.read_csv(BASE_DIR / 'data' / 'tests' / 'fake_fire.csv')
df_raw.head()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,sensor,bright_ti4,bright_ti5,expected
0,-13.110948,-41.253002,320,1,1,2026-07-25,1230,N,VIIRS,n,2.0,300,10,D,VIIRS,330,310,inside
1,-13.157337,-41.174462,320,1,1,2026-07-25,1230,N,VIIRS,n,2.0,300,10,D,VIIRS,330,310,inside
2,-12.703435,-41.466544,320,1,1,2026-07-25,1230,N,VIIRS,n,2.0,300,10,D,VIIRS,330,310,inside
3,-13.207426,-41.242774,320,1,1,2026-07-25,1230,N,VIIRS,n,2.0,300,10,D,VIIRS,330,310,inside
4,-12.702054,-41.366369,320,1,1,2026-07-25,1230,N,VIIRS,n,2.0,300,10,D,VIIRS,330,310,inside


In [12]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=9)
fire_coord = [list(queimada) for queimada in zip(df['latitude'], df['longitude'])]

for idx, row in df_raw.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Data: {row['acq_date']}<br>"
              f"Sensor: {row['sensor']}<br>"
              f"Confiança: {row['confidence']}",
        icon=folium.Icon(color='red', icon='fire', prefix='fa')
    ).add_to(m)
m

In [13]:
df_filter = filter_parna_chapada(df=df_raw)

Polígono carregado do arquivo KML


In [15]:
# Mapa destacando os pontos de queimadas filtrados pelo polígono do Parque Nacional da Chapada Diamantina
for idx, row in df_filter.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Data: {row['acq_date']}<br>"
              f"Sensor: {row['sensor']}<br>"
              f"Confiança: {row['confidence']}",
        icon=folium.Icon(color='red', icon='fire', prefix='fa')
    ).add_to(m)
m

Aqui podemos perceber que os pontos de queimadas que estão fora do polígono do Parque Nacional da Chapada Diamantina foram filtrados com sucesso, e apenas os pontos dentro do polígono permanecem no DataFrame final.

### 2. Teste de inserção dos dados filtrados no banco de dados PostgreSQL

In [36]:
def load_fire_data(df):
    """
    Insere os dados do DataFrame na tabela fire_detections
    do banco PostgreSQL/PostGIS.

    Args:
        df: DataFrame contendo os dados do FIRMS.
    """

    POSTGRES_USER = os.getenv("POSTGRES_USER", "postgres")
    POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD", "postgres")
    POSTGRES_DATABASE = os.getenv("POSTGRES_DATABASE", "sentinela_chapada")
    POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
    POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")

    DATABASE_URL = (
        f"postgresql+psycopg2://"
        f"{POSTGRES_USER}:{POSTGRES_PASSWORD}"
        f"@{POSTGRES_HOST}:{POSTGRES_PORT}"
        f"/{POSTGRES_DATABASE}"
    )

    engine = create_engine(DATABASE_URL)

    try:
        # Testa a conexão
        with engine.connect() as connection:
            connection.execute(text("SELECT 1"))

        print("Conexão com o PostgreSQL realizada com sucesso!")

        # Insere os dados
        df.to_sql(
            "firms_fire_test",
            con=engine,
            if_exists="append",
            index=False
        )

        print(f"{len(df)} registros inseridos com sucesso!")

    except Exception as e:
        print(f"Erro ao inserir dados: {e}")

    finally:
        engine.dispose()

In [37]:
#df_filter = df_filter.drop(columns='expected')
df_filter["acq_time"] = (
    df_filter["acq_time"]
    .astype(str)
    .str.zfill(4)
)

load_fire_data(df_filter)

Conexão com o PostgreSQL realizada com sucesso!
10 registros inseridos com sucesso!
